
# ECON 425 – Homework 2: Dynamic Asset Pricing (Python/Notebook)

This notebook fetches daily data, builds the dataset, and reproduces parts **A–G**:

- **A**: ADL(1,2) models for three assets vs S&P, with log volume  
- **B**: Compare ADL(1,2) to static CAPM via AIC  
- **C**: Long-Run Propensity (LRP) of S&P → each asset  
- **D**: Granger causality (S&P → asset returns)  
- **E**: Geometric (Koyck) distributed lag for one asset  
- **F**: Merge NASDAQ and plot S&P & NASDAQ levels + NASDAQ return  
- **G**: ADL(1,2) with NASDAQ added, then replaced

> **Assumptions**
> - Assets chosen in HW1: **MLI, CWCO, SEDG**; market indices: S&P 500 (**^GSPC**) and NASDAQ Composite (**^IXIC**).  
> - Date range: last ~13 months to ensure a full 1-year clean sample for modeling.

> **Outputs**
> - Figures and tables saved under `outputs/` next to this notebook.
> - A combined CSV with all variables for reproducibility.


In [1]:

# If you don't have these packages, uncomment the pip cell below.
# %pip install yfinance pandas numpy statsmodels matplotlib pyreadstat

import os, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")

# ------------------- CONFIG (edit if needed) -------------------
ASSETS = ["MLI", "CWCO", "SEDG"]   # Three firms from HW1
SPX_TICKER = "^GSPC"               # S&P 500
NASDAQ_TICKER = "^IXIC"            # NASDAQ Composite (fallback if NASDAQ.dta not provided)
DATE_COL = "daten"
YEAR_BACK_DAYS = 370               # fetch slightly more than 1y, then trim
SAVE_DIR = Path("outputs")
SAVE_DIR.mkdir(exist_ok=True)
DATA_CSV = SAVE_DIR / "econ425_hw2_data.csv"
# ---------------------------------------------------------------

def log_return(series: pd.Series) -> pd.Series:
    return 100.0 * np.log(series).diff()

def safe_log_volume(v: pd.Series) -> pd.Series:
    # natural log of raw volume; guard against 0 with forward/back fill
    v2 = v.replace(0, np.nan)
    v2 = v2.fillna(method="bfill").fillna(method="ffill")
    return np.log(v2)

def add_lags(df, cols, maxlag):
    for c in cols:
        for k in range(1, maxlag+1):
            df[f"{c}_L{k}"] = df[c].shift(k)
    return df


## 1) Fetch daily price & volume (Yahoo Finance) + build dataset

In [2]:

from datetime import datetime, timedelta, timezone
end = datetime.now(timezone.utc)
start = end - timedelta(days=YEAR_BACK_DAYS)

tickers = ASSETS + [SPX_TICKER, NASDAQ_TICKER]
print("Fetching:", tickers, "from", start.date(), "to", end.date())

# Download Adj Close and Volume in one call
raw = yf.download(
    tickers=tickers,
    start=start.strftime("%Y-%m-%d"),
    end=end.strftime("%Y-%m-%d"),
    interval="1d",
    auto_adjust=False,
    threads=True,
    group_by="ticker",
    progress=False,
)

# Normalize to a tidy DataFrame with columns we need
frames = []
for t in tickers:
    if t not in raw.columns.get_level_values(0):
        raise ValueError(f"Ticker {t} not found in downloaded data. Check symbol spelling.")
    sub = raw[t][["Adj Close","Volume"]].copy()
    sub.columns = [f"{t}_adj_close", f"{t}_volume"]
    frames.append(sub)

wide = pd.concat(frames, axis=1)
wide.index.name = DATE_COL

# Ensure no duplicate dates, sort index
wide = wide[~wide.index.duplicated(keep="first")].sort_index()

# Build returns & log-volumes for chosen assets + market
df = wide.copy()
df[f"{SPX_TICKER}_ret"] = log_return(df[f"{SPX_TICKER}_adj_close"])
for t in ASSETS:
    df[f"{t}_ret"] = log_return(df[f"{t}_adj_close"])
    df[f"{t}_lv"]  = safe_log_volume(df[f"{t}_volume"])

# NASDAQ level and return for later
df["nasdaq_level"] = df[f"{NASDAQ_TICKER}_adj_close"]
df["nas_ret"] = log_return(df["nasdaq_level"])

# Reset index to have a clean date column
df = df.reset_index()

# Save raw working dataset
df.to_csv(DATA_CSV, index=False)
print("Saved dataset to:", DATA_CSV)
df.tail(3)


Fetching: ['MLI', 'CWCO', 'SEDG', '^GSPC', '^IXIC'] from 2024-08-28 to 2025-09-02
Saved dataset to: outputs/econ425_hw2_data.csv


,daten,MLI_adj_close,MLI_volume,CWCO_adj_close,CWCO_volume,SEDG_adj_close,SEDG_volume,^GSPC_adj_close,^GSPC_volume,^IXIC_adj_close,^IXIC_volume,^GSPC_ret,MLI_ret,MLI_lv,CWCO_ret,CWCO_lv,SEDG_ret,SEDG_lv,nasdaq_level,nas_ret
249,2025-08-27,96.540001,989500,33.380001,61500,33.09,2711500,6481.399902,4143680000,21590.140625,8040510000,0.238813,0.592178,13.804955,-0.478183,11.026792,2.571308,14.813013,21590.140625,0.212689
250,2025-08-28,96.500000,475200,33.599998,72700,33.25,2056300,6501.859863,4283760000,21705.160156,7807080000,0.315175,-0.041443,13.071491,0.656907,11.194097,0.482364,14.536419,21705.160156,0.531327
251,2025-08-29,95.940002,690100,33.270000,71100,33.82,3097600,6460.259766,4234840000,21455.550781,7715430000,-0.641874,-0.581999,13.444592,-0.986992,11.171843,1.699757,14.946138,21455.550781,-1.156664


## 2) Restrict to most recent ~1 year with full data across needed columns

In [3]:

need_cols = [f"{SPX_TICKER}_ret", "nasdaq_level", "nas_ret"]
for t in ASSETS:
    need_cols += [f"{t}_ret", f"{t}_lv", f"{t}_adj_close", f"{t}_volume"]

df_yr = df.dropna(subset=need_cols).copy()
# Keep last ~260 rows (trading days); adjust if needed
df_yr = df_yr.iloc[-260:].reset_index(drop=True)

# Show sample window
print(df_yr[DATE_COL].min(), "→", df_yr[DATE_COL].max(), f"({len(df_yr)} obs)")
df_yr.head(3)


2024-08-29 00:00:00 → 2025-08-29 00:00:00 (251 obs)


,daten,MLI_adj_close,MLI_volume,CWCO_adj_close,CWCO_volume,SEDG_adj_close,SEDG_volume,^GSPC_adj_close,^GSPC_volume,^IXIC_adj_close,^IXIC_volume,^GSPC_ret,MLI_ret,MLI_lv,CWCO_ret,CWCO_lv,SEDG_ret,SEDG_lv,nasdaq_level,nas_ret
0,2024-08-29,70.510368,453000,27.211115,82800,24.91,1970300,5591.959961,3065640000,17516.429688,5727780000,-0.003938,1.967103,13.023647,0.506705,11.324183,-3.083251,14.493696,17516.429688,-0.225816
1,2024-08-30,71.844292,602000,27.270058,203800,24.33,2080200,5648.399902,4185850000,17713.619141,5531150000,1.004246,1.874139,13.308013,0.216379,12.224894,-2.355917,14.547975,17713.619141,1.119451
2,2024-09-03,67.793114,915400,25.767057,207900,22.07,2904900,5528.930176,3866350000,17136.300781,5813970000,-2.137796,-5.804054,13.727116,-5.669248,12.244812,-9.749095,14.881910,17136.300781,-3.313472


## A) ADL(1,2) for each asset: return ~ return_L1 + S&P (0..2 lags) + log(volume) (0..2 lags)

In [4]:

from collections import defaultdict
results_A = defaultdict(dict)

for t in ASSETS:
    y = f"{t}_ret"
    x_mkt = f"{SPX_TICKER}_ret"
    v = f"{t}_lv"
    tmp = df_yr[[DATE_COL, y, x_mkt, v]].copy()
    tmp = add_lags(tmp, [y, x_mkt, v], 2)
    cols = [f"{y}_L1", x_mkt, f"{x_mkt}_L1", f"{x_mkt}_L2", v, f"{v}_L1", f"{v}_L2"]
    tmp2 = tmp.dropna(subset=[y] + cols).copy()
    X = sm.add_constant(tmp2[cols])
    mod = sm.OLS(tmp2[y], X).fit()
    results_A[t]["model"] = mod
    results_A[t]["data"] = tmp2

    # Save summary
    with open(SAVE_DIR / f"A_ADL12_{t}.txt", "w") as f:
        f.write(mod.summary().as_text())

    # Wald test for dynamic lag response (exclude contemporaneous terms)
    import numpy as np
    Xcols = mod.model.exog_names
    R = np.zeros((4, len(Xcols)))
    name_to_idx = {name: i for i, name in enumerate(Xcols)}
    for r_i, cname in enumerate([f"{x_mkt}_L1", f"{x_mkt}_L2", f"{v}_L1", f"{v}_L2"]):
        R[r_i, name_to_idx[cname]] = 1.0
    wtest = mod.wald_test(R)

    # Ljung–Box on residuals
    lb = acorr_ljungbox(mod.resid, lags=[10], return_df=True)

    # Save diagnostics
    with open(SAVE_DIR / f"A_ADL12_{t}_diagnostics.txt", "w") as f:
        f.write("Wald test (lags of market & volume = 0):\n")
        f.write(str(wtest) + "\n\n")
        f.write("Ljung-Box at lag 10:\n")
        f.write(str(lb) + "\n")

print("Saved ADL(1,2) tables and diagnostics to", SAVE_DIR)


Saved ADL(1,2) tables and diagnostics to outputs


## B) Compare AIC: ADL(1,2) vs static CAPM (same sample)

In [5]:

aic_rows = []
for t in ASSETS:
    y = f"{t}_ret"
    tmp2 = results_A[t]["data"]
    capm = sm.OLS(tmp2[y], sm.add_constant(tmp2[f"{SPX_TICKER}_ret"])).fit()
    adl = results_A[t]["model"]
    aic_rows.append({
        "asset": t,
        "AIC_CAPM": capm.aic,
        "AIC_ADL12": adl.aic,
        "ADL_better": adl.aic < capm.aic
    })
aic_df = pd.DataFrame(aic_rows)
aic_df.to_csv(SAVE_DIR / "B_AIC_compare.csv", index=False)
aic_df


,asset,AIC_CAPM,AIC_ADL12,ADL_better
0,MLI,973.527479,981.290433,False
1,CWCO,1022.911204,1022.419348,True
2,SEDG,1680.077284,1688.161174,False


## C) Long-Run Propensity (LRP) of S&P → asset

In [6]:

def lrp_from_adl(mod, yname, xname):
    b0 = mod.params.get(xname, np.nan)
    b1 = mod.params.get(f"{xname}_L1", np.nan)
    b2 = mod.params.get(f"{xname}_L2", np.nan)
    rho = mod.params.get(f"{yname}_L1", np.nan)
    return (b0 + b1 + b2) / (1 - rho)

lrp_rows = []
for t in ASSETS:
    y = f"{t}_ret"
    mod = results_A[t]["model"]
    LRP = lrp_from_adl(mod, yname=y, xname=f"{SPX_TICKER}_ret")
    lrp_rows.append({"asset": t, "LRP_S&P_to_asset": LRP})
lrp_df = pd.DataFrame(lrp_rows)
lrp_df.to_csv(SAVE_DIR / "C_LRP.csv", index=False)
lrp_df


,asset,LRP_S&P_to_asset
0,MLI,1.049821
1,CWCO,0.331992
2,SEDG,1.642051


## D) Granger causality: Do past S&P lags help predict asset returns?

In [7]:

granger_rows = []
for t in ASSETS:
    mod = results_A[t]["model"]
    Xcols = mod.model.exog_names
    import numpy as np
    R = np.zeros((2, len(Xcols)))
    for i, cname in enumerate(Xcols):
        if cname == f"{SPX_TICKER}_ret_L1":
            R[0, i] = 1.0
        if cname == f"{SPX_TICKER}_ret_L2":
            R[1, i] = 1.0
    res = mod.wald_test(R)
    granger_rows.append({"asset": t, "stat": float(res.statistic), "pvalue": float(res.pvalue)})
granger_df = pd.DataFrame(granger_rows)
granger_df.to_csv(SAVE_DIR / "D_Granger.csv", index=False)
granger_df


,asset,stat,pvalue
0,MLI,0.765890,0.466048
1,CWCO,0.006817,0.993207
2,SEDG,0.756676,0.470335


## E) Geometric (Koyck) Distributed Lag for one asset (first in list)

In [8]:

t = ASSETS[0]
y = f"{t}_ret"
x = f"{SPX_TICKER}_ret"

tmp = df_yr[[DATE_COL, y, x]].copy()
tmp[f"{y}_L1"] = tmp[y].shift(1)
tmp2 = tmp.dropna().copy()

X = sm.add_constant(tmp2[[f"{y}_L1", x]])
koyck = sm.OLS(tmp2[y], X).fit()

lam = koyck.params[f"{y}_L1"]
beta = koyck.params[x]

first_lag = beta * lam
tenth_lag = beta * (lam**10)
lrp_koyck = beta / (1 - lam)

with open(SAVE_DIR / f"E_Koyck_{t}.txt", "w") as f:
    f.write(koyck.summary().as_text())
    f.write("\n\n")
    f.write(str({"lambda": float(lam), "beta": float(beta), "lag1": float(first_lag), "lag10": float(tenth_lag), "LRP": float(lrp_koyck)}))

print({"lambda": float(lam), "beta": float(beta), "lag1": float(first_lag), "lag10": float(tenth_lag), "LRP": float(lrp_koyck)})


{'lambda': 0.004861812947225007, 'beta': 1.2087436023284313, 'lag1': 0.005876685295675762, 'lag10': 8.919009917321729e-24, 'LRP': 1.2146489985559443}


## F) Add NASDAQ; Plots for S&P vs NASDAQ levels and NASDAQ return

In [9]:

# We already have nasdaq_level and spx level in df (full range); make level-indexed plots
levels = df[[DATE_COL, f"{SPX_TICKER}_adj_close", "nasdaq_level"]].dropna().copy()

# Re-base to 100 for a clean comparison
levels2 = levels.copy()
levels2[f"{SPX_TICKER}_idx"] = 100 * levels2[f"{SPX_TICKER}_adj_close"] / levels2[f"{SPX_TICKER}_adj_close"].iloc[0]
levels2["NASDAQ_idx"]         = 100 * levels2["nasdaq_level"] / levels2["nasdaq_level"].iloc[0]

plt.figure()
plt.plot(levels2[DATE_COL], levels2[f"{SPX_TICKER}_idx"], label="S&P (idx=100)")
plt.plot(levels2[DATE_COL], levels2["NASDAQ_idx"], label="NASDAQ (idx=100)")
plt.title("S&P vs NASDAQ (levels, indexed to 100)")
plt.xlabel("Date"); plt.ylabel("Index")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "F_levels_spx_nasdaq.png", dpi=180)
plt.close()

# NASDAQ daily return plot
nas_ret = df[[DATE_COL, "nas_ret"]].dropna().copy()
plt.figure()
plt.plot(nas_ret[DATE_COL], nas_ret["nas_ret"])
plt.title("NASDAQ Daily Log Return (×100)")
plt.xlabel("Date"); plt.ylabel("Return")
plt.tight_layout()
plt.savefig(SAVE_DIR / "F_nasdaq_return.png", dpi=180)
plt.close()

print("Saved figures to:", SAVE_DIR)


Saved figures to: outputs


## G) ADL(1,2) with NASDAQ added (S&P + NASDAQ), and replaced (NASDAQ only)

In [10]:

def fit_adl_with_vars(asset, mkt_vars):
    y = f"{asset}_ret"
    v = f"{asset}_lv"
    tmp = df_yr[[DATE_COL, y, v] + mkt_vars].copy()
    tmp = add_lags(tmp, [y] + mkt_vars + [v], 2)
    cols = [f"{y}_L1"] + sum([[m, f"{m}_L1", f"{m}_L2"] for m in mkt_vars], []) + [v, f"{v}_L1", f"{v}_L2"]
    tmp2 = tmp.dropna(subset=[y] + cols).copy()
    X = sm.add_constant(tmp2[cols])
    mod = sm.OLS(tmp2[y], X).fit()
    return mod

comp_rows = []
for t in ASSETS:
    base = results_A[t]["model"]
    add  = fit_adl_with_vars(t, [f"{SPX_TICKER}_ret", "nas_ret"])
    rep  = fit_adl_with_vars(t, ["nas_ret"])

    comp_rows.append({
        "asset": t,
        "AIC_base_ADL(SPX)": base.aic,
        "AIC_add_NAS(SPX+NAS)": add.aic,
        "AIC_replace_NAS(NAS only)": rep.aic
    })

    # Save tables
    with open(SAVE_DIR / f"G_ADL_add_{t}.txt", "w") as f: f.write(add.summary().as_text())
    with open(SAVE_DIR / f"G_ADL_rep_{t}.txt", "w") as f: f.write(rep.summary().as_text())

comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(SAVE_DIR / "G_AIC_compare.csv", index=False)
comp_df


,asset,AIC_base_ADL(SPX),AIC_add_NAS(SPX+NAS),AIC_replace_NAS(NAS only)
0,MLI,981.290433,981.642703,999.872492
1,CWCO,1022.419348,1020.392494,1026.962768
2,SEDG,1688.161174,1693.046091,1691.151255



---

## Write‑up prompts (paste into your HW document)

- **(A)** For each asset, paste the ADL(1,2) table and briefly discuss:  
  i) whether lagged market/volume are jointly significant (Wald test)  
  ii) whether residuals show serial correlation (Ljung–Box) and any hint of cycles

- **(B)** Compare AIC of ADL vs CAPM; say which model is better per asset.

- **(C)** Report **LRP** of S&P→asset and interpret (long‑run total effect).

- **(D)** Report Granger Wald statistics and p‑values. Conclude if S&P Granger‑causes the asset.

- **(E)** Report Koyck **lambda**, implied lag‑1/lag‑10 effects, and LRP. Comment on speed of decay.

- **(F)** Insert the two figures: S&P vs NASDAQ levels, and NASDAQ return.

- **(G)** Compare AICs across: base (S&P), added (S&P+NASDAQ), replaced (NASDAQ only). Discuss whether NASDAQ adds information and for which assets.

**All figures/tables saved in** `outputs/`.
